# Jev as a Judge

A doctor's scheduling assistant must turn messages such as “I can work late on Friday”
into the right action. This lab checks whether it understood the request, using Jev and GPT-5.

We compare three approaches: **Jev alone**, **GPT-5 alone**, and **Jev with GPT-5 fallback**.
The notebook makes live API calls and records the verdict, cost and any escalation.

## What are we judging?

The scheduling assistant reads a doctor's message and chooses an action. The judge sees
that message and the predicted action; it does not execute appointments or see the human answer.

**Failure category:** Failure to interpret indirect or underspecified requests.

**Binary question:** Did the router fail to interpret the indirect or underspecified scheduling request?

### The routing policy

| Intent | What the assistant can do |
| --- | --- |
| `LIST_APPOINTMENTS` | View existing appointments; not analyze historical patient counts. |
| `CANCEL_APPOINTMENTS` | Cancel all appointments in a date/time range. |
| `RESCHEDULE_APPOINTMENTS` | Move all appointments from one date/time range to another. |
| `BLOCK_SLOTS` | Make the doctor unavailable. |
| `UNBLOCK_SLOTS` | Restore availability. |
| `UPDATE_SPECIAL_SLOTS` | Add, extend or reduce working hours. |
| `CONVERSATIONAL_GREETING` | Respond to an opening social greeting. |
| `CLOSING_CONVERSATION` | Recognize a closing thanks, acknowledgment or goodbye. |
| `UNSUPPORTED_REQUEST` | A scheduling operation outside these capabilities, or a request whose operation remains unclear. |
| `IRRELEVANT_QUERY` | A request unrelated to the doctor, schedule, patients or hospital. |

Creating an appointment, or canceling/moving a specific appointment identified by patient
name or relative position, is unsupported. Missing a date, time or destination does **not**
make a recognizable supported action unsupported. When wording or missing calendar state
leaves multiple reasonable interpretations, accept a defensible prediction. Do not invent
prior blocks, working hours, or a rule for prioritizing multiple actions.

### The TRUE/FALSE rule

Return **TRUE** only when all three are established:

1. The message expresses the action indirectly, through shorthand or a misspelling, or with incomplete execution details. Ordinary politeness alone is insufficient.
2. The predicted intent is ruled out by every reasonable reading allowed by the policy.
3. The mismatch is a failure to recover that scheduling meaning.

Return **FALSE** when this failure is not established. FALSE includes correct routes,
defensible ambiguity, and different errors such as reversing explicit available/unavailable
wording or crossing an unsupported capability boundary. It does not mean “correct overall.”

Both Jev and GPT-5 receive these rules and the same six worked examples below.

| Doctor's message | Router prediction | Judge | Why |
| --- | --- | --- | --- |
| I can work a half-day this Saturday. | `UNSUPPORTED_REQUEST` | **TRUE** | Working availability is recoverable despite missing hours. |
| I need to reschedule. | `RESCHEDULE_APPOINTMENTS` | **FALSE** | The rescheduling action was correctly recognized. |
| I can work late on the 3rd until 8 PM. | `UNSUPPORTED_REQUEST` | **TRUE** | The implied extension of working hours was missed. |
| I'm not working *this* Saturday. | `UNBLOCK_SLOTS` | **FALSE** | This is an explicit polarity error, a different category. |
| Move my afternoon. | `UNSUPPORTED_REQUEST` | **TRUE** | Recognizable rescheduling shorthand was rejected. |
| Move Mr. Johnson's appointment from Monday to Tuesday. | `UNSUPPORTED_REQUEST` | **FALSE** | Moving a named patient's appointment is unsupported. |

## 1. Connect to the live APIs

Run the cells in order with a TypeSafe API key and an OpenAI API key.
Keys can come from environment variables, Colab Secrets, an ignored `.env`, or hidden input.
Every judge invocation makes a fresh request; running an inference cell again uses the APIs again.
GPT-5 uses medium reasoning effort and a structured Boolean answer with a brief explanation.

In [ ]:
import os
import subprocess
import sys
from pathlib import Path

# Colab needs a checkout; locally, open this notebook inside the repository.
try:
    import google.colab
    IN_COLAB = True
except ImportError:
    IN_COLAB = False
if IN_COLAB:
    checkout = Path("/content/routing-evals-lab")
    if not checkout.exists():
        subprocess.run(["git", "clone", "--depth", "1", "--branch", "codex/jev-routing-judge",
                        "https://github.com/The-Gen-Academy/Mastering-Agentic-AI-Week4-Session1.git",
                        str(checkout)], check=True)
    os.chdir(checkout)
    subprocess.run([sys.executable, "-m", "pip", "install", "-q",
                    "pandas", "python-dotenv"], check=True)

folders = [Path.cwd(), Path("routing_agent_evals"), Path("AI-Evals-Principles/routing_agent_evals")]
LAB = next(folder.resolve() for folder in folders if (folder / "judge_lab.py").exists())
os.chdir(LAB)
sys.path.insert(0, str(LAB))

In [ ]:
import pandas as pd
from IPython.display import display
from prompts.llm_as_judge_prompt import (
    FAILURE_CATEGORY, FAILURE_QUESTION, JEV_JUDGE_PROMPT,
    INDIRECT_OR_UNDERSPECIFIED_FAILURE_PROMPT,
)
from judge_lab import (
    load_dataset, request_for, batch, needs_llm, run_cascade, resolved,
    comparison_metrics, always_false, confidence_bands, threshold_sweep, save_json,
)
from notebook_utils import setup_run, save_run

THRESHOLD = 0.70  # Starting setting; compare alternatives on practice cases below.
client, RUN = setup_run(LAB, prompts={
    "jev": JEV_JUDGE_PROMPT,
    "llm": INDIRECT_OR_UNDERSPECIFIED_FAILURE_PROMPT,
})

## 2. Load the labeled examples

We use **21 practice cases** to examine confidence thresholds, then **150 different cases**
to evaluate the judges. Separating these steps prevents us from choosing a cutoff and
claiming success on the very same examples. Both sets run automatically in this notebook.

The six worked examples above are excluded from scoring. Human TRUE/FALSE labels are
used to calculate results; neither judge receives them in its input.

In [ ]:
calibration_rows = load_dataset(LAB / "data", split="calibration")
rows = load_dataset(LAB / "data", split="validation")
assert {r["id"] for r in calibration_rows}.isdisjoint(r["id"] for r in rows)
display(pd.DataFrame([
    {"purpose": name, "cases": len(part), "TRUE labels": sum(r["human_label"] == "TRUE" for r in part)}
    for name, part in [("Check thresholds", calibration_rows), ("Evaluate judges", rows)]
]))
print("Example sent to both judges:", request_for(rows[0], "jev")["state"])

## 3. How to choose the confidence threshold

A higher cutoff sends more cases to GPT-5. A lower cutoff accepts more Jev decisions,
which can reduce cost but also keep confident mistakes. Confidence measures how strongly
Jev prefers its answer; it is not a measured percentage of correct decisions.

For binary Choice, `confidence = 2 × max(P(TRUE), P(FALSE)) − 1`.
Thus confidence **0.70** means the winning option has probability **0.85**, whether that
option is TRUE or FALSE. [TypeSafe's definition](https://docs.typesafe.ai/confidence).

The [CMU Jev-as-a-Judge study](https://arxiv.org/html/2609.26550v5) tested thresholds on
locally labeled examples. It preferred the threshold accepting the most Jev decisions
while a one-sided 95% confidence bound kept estimated accuracy loss versus the fallback
within two percentage points. It then tested the chosen threshold on separate examples.
The paper uses the winning probability, so its thresholds must be converted before using
this notebook's native-confidence gate; it does not prescribe a universal 0.70 cutoff.

The table below compares candidate cutoffs on our 21 practice cases, including 0.70.
`within_2pp_bound` shows the study's conservative selection check. With only 21 cases,
this is a small diagnostic; inspect failure recall and false alarms as well as accuracy.
Choose the cost/quality tradeoff before examining evaluation results. This run uses the
starting `THRESHOLD = 0.70`; the table does not silently change it.

In [ ]:
jev_practice = batch(calibration_rows, lambda row: client.call(row, "jev"))
llm_practice = batch(calibration_rows, lambda row: client.call(row, "llm"))
# Candidate native-confidence cutoffs; 0.70 is the starting value for this run.
calibration = threshold_sweep(calibration_rows, jev_practice, llm_practice,
                              thresholds=[0.0, 0.2, 0.4, 0.6, 0.7, 0.8, 0.9, 0.98, 1.0])
threshold_table = pd.DataFrame(calibration).rename(columns={"noninferiority_pass": "within_2pp_bound"})
display(threshold_table[["threshold", "accepted_fraction", "cascade_accuracy", "precision_TRUE",
                         "recall_TRUE", "FP", "FN", "cost_savings_fraction", "within_2pp_bound", "status"]].round(4))
save_json(RUN / "threshold_diagnostics.json", calibration)
print("Cutoff used for evaluation:", THRESHOLD)

## 4. Run Jev, then ask GPT-5 when needed

**Jev call → validate output → confidence gate → GPT-5 fallback → final verdict → logging.**

Keep a valid Jev answer at native confidence **0.70 or above**, for either TRUE or FALSE.
Below the cutoff, or when Jev's output is invalid, ask GPT-5 the same question with the
same policy and evidence. GPT-5 sees no Jev answer or confidence. If fallback fails too,
record an unresolved verdict instead of inventing TRUE or FALSE.

In [ ]:
jev = batch(rows, lambda row: client.call(row, "jev"))
example_id = rows[0]["id"]
display(pd.DataFrame([jev[example_id]]).reindex(columns=["label", "confidence", "probabilities", "error"]))

In [ ]:
before_fallback = client.new_requests
cascade = run_cascade(rows, client, THRESHOLD, jev=jev)
escalated_ids = {r["id"] for r in rows if needs_llm(jev[r["id"]], THRESHOLD)}
assert {key for key, value in cascade.items() if value["route"] == "LLM"} == escalated_ids
print(f"GPT-5 fallback used for {len(escalated_ids)}/{len(rows)} cases.")
print("Live HTTP attempts during fallback:", client.new_requests - before_fallback)

## 5. Run GPT-5 alone

This is a separate live baseline: GPT-5 judges every evaluation case, including cases
that already went through fallback. It receives the same question, policy, examples
and structured output format. Responses are not reused from the fallback run.

In [ ]:
before_baseline = client.new_requests
llm = batch(rows, lambda row: client.call(row, "llm"))
print("Fresh GPT-5 baseline HTTP attempts:", client.new_requests - before_baseline)
display(pd.DataFrame([llm[example_id]]).reindex(columns=["label", "reason", "error"]))

## 6. Compare the three approaches

Score the same successfully judged cases across all methods. Also report each method's
coverage and unresolved count so missing answers stay visible. TRUE recall shows how
many selected failures were found; precision shows how many flags were justified.
Compare both with the simple baseline that always answers FALSE.

Costs include all attempts. Each method's cost describes running that method on the
150 evaluation cases. The full notebook additionally pays for practice calls and the
separate GPT-5 baseline. Charges may be token-price estimates; missing costs stay unknown.

In [ ]:
approaches = {"Always FALSE": always_false(rows), "Jev": jev, "GPT-5": llm, "Jev + GPT-5": cascade}
scored_ids = [r["id"] for r in rows if all(resolved(a[r["id"]]) for a in approaches.values())]
comparison = pd.DataFrame(comparison_metrics(rows, approaches))
comparison["cost_per_1000_USD"] = comparison.cost_usd / len(rows) * 1000
comparison["GPT5_fraction"] = comparison.llm_routes / len(rows)
comparison.loc[comparison.method.eq("GPT-5"), "GPT5_fraction"] = 1.0
columns = ["method", "scored_n", "coverage", "accuracy", "precision_TRUE", "recall_TRUE", "f1_TRUE",
           "FP", "FN", "unresolved", "GPT5_fraction", "cost_per_1000_USD"]
table = comparison[columns].copy()
table.loc[comparison.unknown_cost_attempts.gt(0), "cost_per_1000_USD"] = float("nan")
display(table.round(4))
# Avoid double-counting Jev when reporting what this whole notebook spent.
full_cost = sum(r["cost_usd"] for group in (jev_practice, llm_practice, cascade, llm) for r in group.values())
full_unknown = sum(r["unknown_cost_attempts"] for group in (jev_practice, llm_practice, cascade, llm) for r in group.values())
print("Full notebook cost (reported + estimated USD):", round(full_cost, 6))
print("Attempts with unknown cost:", full_unknown)

## 7. Inspect errors and save the run

Review the doctor's message, predicted intent, verdict and human label together.
The GPT-5 baseline and fallback are independent calls, so their answers can differ.
Look especially for confident Jev errors that bypass fallback. Labels in this exercise
are provisional; disagreements deserve inspection, not automatic relabeling.

Each API call is logged separately. The decision table records the chosen route,
confidence, final verdict, errors and call IDs. Logs are for inspection and are never
read back to answer later requests.

In [ ]:
bands = confidence_bands(rows, jev, llm, THRESHOLD)
display(pd.DataFrame(bands))
records = []
for row in rows:
    case_id = row["id"]
    j, l, final = jev[case_id], llm[case_id], cascade[case_id]
    gate_reason = ("invalid_jev_output" if j.get("error") else
                   "below_confidence_threshold" if case_id in escalated_ids else
                   "confidence_at_or_above_threshold")
    records.append({
        "ID": case_id, "category": FAILURE_CATEGORY, "question": FAILURE_QUESTION,
        "request": row["user_query"], "predicted_intent": row["predicted_intent"],
        "human": row["human_label"], "included_in_paired_scores": case_id in scored_ids,
        "Jev": j["label"], "Jev_confidence": j.get("confidence"), "threshold": THRESHOLD,
        "gate_reason": gate_reason, "route": final["route"], "GPT5_baseline": l["label"],
        "GPT5_baseline_reason": l.get("reason"), "final": final["label"],
        "fallback_reason": final.get("reason") if final["route"] == "LLM" else None,
        "Jev_error": j.get("error"), "GPT5_baseline_error": l.get("error"), "final_error": final.get("error"),
        "Jev_call_id": j["call_id"], "GPT5_baseline_call_id": l["call_id"], "final_call_id": final["call_id"],
        "cascade_cost_usd": final["cost_usd"], "unknown_cost_attempts": final["unknown_cost_attempts"],
    })
review = pd.DataFrame(records)
disagreements = review[review.Jev.ne(review.GPT5_baseline) | review.final.ne(review.human)]
display(disagreements.head(10))
review.to_csv(RUN / "all_results.csv", index=False)
disagreements.to_csv(RUN / "disagreements.csv", index=False)
save_run(client, RUN, rows, THRESHOLD, comparison, bands, scored_ids,
         calibration_rows=calibration_rows, calibration=calibration)
save_json(RUN / "total_usage.json", {"cost_usd": full_cost, "unknown_cost_attempts": full_unknown,
                                    "http_attempts": client.new_requests})